In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

In [ ]:
!pip install -q kaggle
!pip install -q torchvision matplotlib


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("briscdataset/brisc2025")

print("Path to dataset files:", path)

In [ ]:
import os

# List the contents of the downloaded dataset directory
print(f"Contents of {path}:")
for root, dirs, files in os.walk(path):
    level = root.replace(path, '').count(os.sep)
    indent = ' ' * 4 * (level)
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 4 * (level + 1)
    for f in files:
        print(f'{subindent}{f}')
    if level == 4: # Explore one more level deep to see contents of train/
        break

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import random
import os

# Construct the base path to the training images
base_train_path = os.path.join(path, 'brisc2025', 'classification_task', 'train')

# Get a list of all class directories
class_dirs = [d for d in os.listdir(base_train_path) if os.path.isdir(os.path.join(base_train_path, d))]

if not class_dirs:
    print(f"No class directories found in {base_train_path}")
else:
    print(f"Found classes: {class_dirs}")

    sample_images = []
    # Try to collect 15 images, picking from available classes
    for _ in range(15):
        if not class_dirs:
            break
        # Randomly pick a class
        chosen_class = random.choice(class_dirs)
        class_path = os.path.join(base_train_path, chosen_class)

        # Get all image files in the chosen class directory
        images_in_class = [f for f in os.listdir(class_path) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp'))]

        if images_in_class:
            # Pick a random image from this class
            chosen_image_name = random.choice(images_in_class)
            sample_images.append(os.path.join(class_path, chosen_image_name))
        else:
            # If a class has no images, remove it from consideration for future picks
            class_dirs.remove(chosen_class)

    if not sample_images:
        print("Could not find any images to display.")
    else:
        # Display images in a 3x5 grid
        fig, axes = plt.subplots(3, 5, figsize=(15, 9))
        axes = axes.flatten()

        for i, img_path in enumerate(sample_images):
            if i >= len(axes):
                break
            try:
                img = Image.open(img_path)
                axes[i].imshow(img)
                axes[i].set_title(os.path.basename(os.path.dirname(img_path)))
                axes[i].axis('off')
            except Exception as e:
                axes[i].set_title(f"Error loading {os.path.basename(img_path)}")
                axes[i].axis('off')
                print(f"Error loading image {img_path}: {e}")

        # Hide any unused subplots
        for j in range(i + 1, len(axes)):
            fig.delaxes(axes[j])

        plt.tight_layout()
        plt.show()

In [ ]:
import torch
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, random_split
from PIL import Image
import os

# Define the custom dataset for classification
class BrainTumorDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []
        self.class_to_idx = {}

        # Collect all image paths and assign labels
        for i, class_name in enumerate(sorted(os.listdir(root_dir))):
            class_path = os.path.join(root_dir, class_name)
            if os.path.isdir(class_path):
                self.class_to_idx[class_name] = i
                for img_name in os.listdir(class_path):
                    if img_name.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp')):
                        self.image_paths.append(os.path.join(class_path, img_name))
                        self.labels.append(i)
        print(f"Found {len(self.image_paths)} images belonging to {len(self.class_to_idx)} classes: {self.class_to_idx}")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, label

# Define transformations
# These are basic transforms, can be expanded with more augmentation
transform = transforms.Compose([
    transforms.Resize((224, 224)), # Resize images to a common size
    transforms.ToTensor(),        # Convert images to PyTorch tensors
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # Normalize
])

# Instantiate the dataset
# `base_train_path` is already defined from previous steps: /root/.cache/kagglehub/datasets/briscdataset/brisc2025/versions/6/brisc2025/classification_task/train
dataset = BrainTumorDataset(root_dir=base_train_path, transform=transform)

# Split the dataset into training and validation sets
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

print(f"Training set size: {len(train_dataset)}")
print(f"Validation set size: {len(val_dataset)}")

# Create DataLoaders
batch_size = 32 # Can be adjusted based on GPU memory
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

print("DataLoaders created successfully.")

In [ ]:
import torch.nn as nn
import timm # PyTorch Image Models library for ViT

# Check if CUDA is available and set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Number of output classes for classification
num_classes = len(dataset.class_to_idx)
print(f"Number of classification classes: {num_classes}")

class DiTMultiTask(nn.Module):
    def __init__(self, num_classes, segmentation_output_channels=1, img_size=(224, 224), patch_size=16, embed_dim=768, depth=12, num_heads=12):
        super().__init__()
        # Using a pre-trained Vision Transformer (ViT) as the backbone
        # For a full DiT, this backbone would also incorporate timestep embeddings and noise prediction
        # Here, it's simplified to a standard ViT for feature extraction.
        self.vit_backbone = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=0) # num_classes=0 removes the default classification head

        # Classification Head
        # The ViT outputs features from the [CLS] token (or pooled features), which we'll use for classification
        self.classification_head = nn.Linear(self.vit_backbone.num_features, num_classes)

        # Segmentation Head (Placeholder/Simplified)
        # For actual segmentation, a decoder structure is needed to upsample features
        # to the original image resolution. This is a very simplified example.
        # If you have segmentation masks, this part would be more complex, perhaps a U-Net style decoder.
        self.segmentation_head = nn.Sequential(
            nn.ConvTranspose2d(self.vit_backbone.num_features, embed_dim // 2, kernel_size=patch_size, stride=patch_size), # Upsample to a larger feature map
            nn.ReLU(),
            nn.ConvTranspose2d(embed_dim // 2, segmentation_output_channels, kernel_size=patch_size, stride=patch_size) # Further upsample to image resolution
            # This simple structure will likely produce low-quality masks. A proper segmentation head
            # would involve multiple upsampling layers and skip connections from the encoder.
        )

        # Adjust ViT to output features for segmentation (if needed)
        # By default, timm ViT outputs a flattened feature vector from the [CLS] token.
        # To get spatial features for segmentation, we need to extract patched features.
        # This requires modifying how ViT's forward pass is typically used or directly accessing intermediate layers.
        # For simplicity, we'll reshape the patch embeddings to a spatial grid before feeding to the segmentation head.
        self.patch_size = patch_size
        self.img_size = img_size


    def forward(self, x):
        batch_size = x.shape[0]
        # Pass through ViT backbone
        # Extract features before the classification head, typically the [CLS] token or pooled features
        # timm's ViT can return patch embeddings and CLS token if configured.

        # This approach gets the features from the ViT's global pooling (cls token)
        # For segmentation, we need to work with the patch embeddings.
        features = self.vit_backbone.forward_features(x) # [CLS] token + patch embeddings

        # Classification output comes from the [CLS] token
        cls_token_features = features[:, 0] # First token is the [CLS] token
        classification_output = self.classification_head(cls_token_features)

        # Segmentation output
        # Take the patch embeddings (excluding CLS token) and reshape to a 2D grid
        patch_features = features[:, 1:] # All tokens except the [CLS] token

        # Calculate grid dimensions
        num_patches_h = self.img_size[0] // self.patch_size
        num_patches_w = self.img_size[1] // self.patch_size

        # Reshape to (batch_size, embed_dim, num_patches_h, num_patches_w)
        spatial_features = patch_features.permute(0, 2, 1).view(batch_size, self.vit_backbone.embed_dim, num_patches_h, num_patches_w)
        segmentation_output = self.segmentation_head(spatial_features)

        # Ensure segmentation_output has the same size as input image (C, H, W)
        # This might require interpolation/cropping if the simple ConvTranspose doesn't match perfectly.
        # For this demonstration, we'll rely on the ConvTranspose sizes.

        return classification_output, segmentation_output

# Instantiate the model and move to device
# Assuming a single output channel for binary segmentation mask (e.g., tumor vs. background)
model = DiTMultiTask(num_classes=num_classes, segmentation_output_channels=1).to(device)

print("DiT-MultiTask model initialized.")
print(model)

# Define loss functions
# For classification: CrossEntropyLoss
classification_criterion = nn.CrossEntropyLoss()

# For segmentation: BCEWithLogitsLoss (suitable for binary segmentation like tumor/no-tumor)
# Note: This will need ground truth masks for training.
segmentation_criterion = nn.BCEWithLogitsLoss()

# Define optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

print("Loss functions and optimizer initialized.")

In [ ]:
import torch.nn as nn
import timm # PyTorch Image Models library for ViT
import torch
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, random_split
from PIL import Image
import os
import gc # Import garbage collector
import torch.amp # Import for Automatic Mixed Precision
from tqdm import tqdm # Import tqdm for progress bars
import torch.nn.functional as F
import numpy as np
from scipy.ndimage import distance_transform_edt

# Define custom loss functions here
class TverskyLoss(nn.Module):
    def __init__(self, alpha: float = 0.3, beta: float = 0.7, epsilon: float = 1e-6):
        super(TverskyLoss, self).__init__()
        self.alpha = alpha
        self.beta = beta
        self.epsilon = epsilon

    def forward(self, y_pred: torch.Tensor, y_true: torch.Tensor) -> torch.Tensor:
        # Apply sigmoid to predictions to get probabilities
        y_pred = torch.sigmoid(y_pred)

        # Flatten tensors for easier calculation
        y_pred = y_pred.view(-1)
        y_true = y_true.view(-1)

        # Calculate True Positives, False Positives, False Negatives
        true_positives = (y_pred * y_true).sum()
        false_positives = ((1 - y_true) * y_pred).sum()
        false_negatives = (y_true * (1 - y_pred)).sum()

        # Calculate Tversky Index
        # Formula: (TP + epsilon) / (TP + alpha * FN + beta * FP + epsilon)
        numerator = true_positives + self.epsilon
        denominator = true_positives + self.alpha * false_negatives + self.beta * false_positives + self.epsilon

        tversky_index = numerator / denominator
        tversky_loss = 1.0 - tversky_index

        return tversky_loss

class FocalLoss(nn.Module):
    def __init__(self, alpha: float = 0.25, gamma: float = 2.0, epsilon: float = 1e-6):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon

    def forward(self, y_pred_logits: torch.Tensor, y_true: torch.Tensor) -> torch.Tensor:
        # Flatten tensors for easier calculation
        y_pred_logits_flat = y_pred_logits.view(-1)
        y_true_flat = y_true.view(-1)

        # Calculate probabilities from logits for pt calculation
        y_pred_proba = torch.sigmoid(y_pred_logits_flat)

        # Calculate pt
        pt = torch.where(y_true_flat == 1, y_pred_proba, 1 - y_pred_proba)

        # Clip pt to avoid log(0) issues
        pt = torch.clamp(pt, self.epsilon, 1. - self.epsilon)

        # Calculate binary cross entropy with logits for each pixel
        bce = F.binary_cross_entropy_with_logits(y_pred_logits_flat, y_true_flat, reduction='none')

        # Apply alpha weighting
        alpha_t = torch.where(y_true_flat == 1, self.alpha, 1 - self.alpha)

        focal_loss = -alpha_t * ((1 - pt) ** self.gamma) * bce

        return focal_loss.mean()

class BoundaryLoss(nn.Module):
    def __init__(self, scale_factor: float = 10.0):
        super(BoundaryLoss, self).__init__()
        self.scale_factor = scale_factor

    def forward(self, y_pred_logits: torch.Tensor, y_true: torch.Tensor) -> torch.Tensor:
        y_true_float = y_true.float()
        device = y_pred_logits.device
        batch_abs_sdms = []

        for i in range(y_true_float.shape[0]):
            mask_np = y_true_float[i, 0].cpu().numpy()
            dist_to_background = distance_transform_edt(mask_np)
            dist_to_foreground = distance_transform_edt(1 - mask_np)
            sdm_np = dist_to_background - dist_to_foreground
            abs_sdm_np = np.abs(sdm_np)
            batch_abs_sdms.append(torch.from_numpy(abs_sdm_np).float().unsqueeze(0).to(device))

        abs_sdm_true = torch.stack(batch_abs_sdms, dim=0)

        y_pred_logits = y_pred_logits.squeeze(1)
        y_true_float = y_true_float.squeeze(1)
        abs_sdm_true = abs_sdm_true.squeeze(1)

        weight_map = 1.0 + torch.exp(-abs_sdm_true / self.scale_factor)
        bce_per_pixel = F.binary_cross_entropy_with_logits(y_pred_logits, y_true_float, reduction='none')
        weighted_bce_loss = bce_per_pixel * weight_map
        boundary_loss = weighted_bce_loss.mean()

        return boundary_loss

class CombinedSegmentationLoss(nn.Module):
    def __init__(self,
                 tversky_loss_fn: TverskyLoss,
                 focal_loss_fn: FocalLoss,
                 boundary_loss_fn: BoundaryLoss,
                 tversky_weight: float = 1.0,
                 focal_weight: float = 1.0,
                 boundary_weight: float = 1.0):
        super(CombinedSegmentationLoss, self).__init__()
        self.tversky_loss_fn = tversky_loss_fn
        self.focal_loss_fn = focal_loss_fn
        self.boundary_loss_fn = boundary_loss_fn
        self.tversky_weight = tversky_weight
        self.focal_weight = focal_weight
        self.boundary_weight = boundary_weight

    def forward(self, y_pred_logits: torch.Tensor, y_true: torch.Tensor) -> torch.Tensor:
        y_true_float = y_true.float()

        loss_tversky = self.tversky_loss_fn(y_pred_logits, y_true_float)
        loss_focal = self.focal_loss_fn(y_pred_logits, y_true_float)
        loss_boundary = self.boundary_loss_fn(y_pred_logits, y_true_float)

        combined_loss = (
            self.tversky_weight * loss_tversky +
            self.focal_weight * loss_focal +
            self.boundary_weight * loss_boundary
        )

        return combined_loss

# Check if CUDA is available and set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Define the custom dataset for classification AND segmentation
class BrainTumorDataset(Dataset):
    def __init__(self, root_dir, mask_root_dir, transform=None, mask_transform=None):
        self.root_dir = root_dir
        self.mask_root_dir = mask_root_dir
        self.transform = transform
        self.mask_transform = mask_transform
        self.image_paths = []
        self.mask_paths = [] # Store mask paths
        self.labels = []
        self.class_to_idx = {}

        # Collect all image paths, corresponding mask paths, and assign labels
        for i, class_name in enumerate(sorted(os.listdir(root_dir))):
            class_path = os.path.join(root_dir, class_name)
            if os.path.isdir(class_path):
                self.class_to_idx[class_name] = i
                for img_name in os.listdir(class_path):
                    if img_name.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp')):
                        image_full_path = os.path.join(class_path, img_name)

                        # Correctly construct mask path:
                        # Take the base name of the image and append .png extension for the mask
                        base_img_name = os.path.splitext(img_name)[0]
                        mask_name = base_img_name + '.png'
                        mask_full_path = os.path.join(mask_root_dir, mask_name)

                        if os.path.exists(mask_full_path):
                            self.image_paths.append(image_full_path)
                            self.mask_paths.append(mask_full_path)
                            self.labels.append(i)
                        else:
                            print(f"Warning: No mask found for image {img_name} at {mask_full_path}. Skipping.")

        print(f"Found {len(self.image_paths)} images and masks belonging to {len(self.class_to_idx)} classes: {self.class_to_idx}")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        mask_path = self.mask_paths[idx]
        image = Image.open(img_path).convert('RGB')
        mask = Image.open(mask_path).convert('L') # Load mask as grayscale
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)
        if self.mask_transform:
            mask = self.mask_transform(mask)

        return image, mask, label # Return image, mask, and label

# Define transformations
# Image transforms: Resize, ToTensor, Normalize
image_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Mask transforms: Resize, ToTensor (masks are typically 0 or 1, no normalization)
# Masks should be loaded as grayscale and converted to float tensor.
mask_transform = transforms.Compose([
    transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.NEAREST), # Use NEAREST for masks
    transforms.ToTensor() # Converts to [0,1] float automatically
])


# NOTE: `path` variable (from kagglehub.dataset_download) is available from cell 'vzahEgYWIe4H'.
# Re-evaluate base paths for robustness if this cell is run independently.
if 'path' not in locals():
    import kagglehub
    path = kagglehub.dataset_download("briscdataset/brisc2025")
    print("kagglehub dataset downloaded to ensure 'path' is defined.")

base_train_image_path = os.path.join(path, 'brisc2025', 'classification_task', 'train')
base_train_mask_path = os.path.join(path, 'brisc2025', 'segmentation_task', 'train', 'masks') # New mask path

# Instantiate the dataset
dataset = BrainTumorDataset(
    root_dir=base_train_image_path,
    mask_root_dir=base_train_mask_path,
    transform=image_transform,
    mask_transform=mask_transform
)

# Split the dataset into training and validation sets
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

print(f"Training set size: {len(train_dataset)}")
print(f"Validation set size: {len(val_dataset)}")

# Create DataLoaders
batch_size = 4 # Reduced further to free up GPU memory
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

print("DataLoaders created successfully.")

# Number of output classes for classification
num_classes = len(dataset.class_to_idx)
print(f"Number of classification classes: {num_classes}")

# Define a U-Net style UpBlock for the decoder
class UpBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.up = nn.Sequential(
            nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2), # Upsample by 2
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.up(x)

class DiTMultiTask(nn.Module):
    def __init__(self, num_classes, segmentation_output_channels=1, img_size=(224, 224), patch_size=16):
        super().__init__()
        # Using an even SMALLER pre-trained Vision Transformer (ViT) as the backbone to save memory
        # Now using patch_size=16 for the vit_tiny model
        self.vit_backbone = timm.create_model(f'vit_tiny_patch{patch_size}_224', pretrained=True, num_classes=0)

        # Dynamically get embed_dim from the backbone
        embed_dim = self.vit_backbone.num_features # For vit_tiny, this is typically 192

        # Classification Head
        self.classification_head = nn.Linear(embed_dim, num_classes)

        # Segmentation Head (U-Net style decoder for progressive upsampling)
        # Input to decoder will be (batch, embed_dim, 224/patch_size, 224/patch_size)
        # For patch_size=16, this is (batch, 192, 14, 14)

        # Define output dimensions for each upsampling stage
        decoder_dims = [embed_dim, embed_dim // 2, embed_dim // 4]

        # Adjust upsampling layers for patch_size=16 input (14x14 spatial resolution)
        self.up1 = UpBlock(decoder_dims[0], decoder_dims[1]) # (192, 14, 14) -> (96, 28, 28)
        self.up2 = UpBlock(decoder_dims[1], decoder_dims[2]) # (96, 28, 28) -> (48, 56, 56)
        self.up3 = UpBlock(decoder_dims[2], decoder_dims[2] // 2) # (48, 56, 56) -> (24, 112, 112)
        # Need one more upsampling block to reach 224x224
        self.up4 = UpBlock(decoder_dims[2] // 2, decoder_dims[2] // 4) # (24, 112, 112) -> (12, 224, 224)

        self.final_seg_conv = nn.Conv2d(decoder_dims[2] // 4, segmentation_output_channels, kernel_size=1)

        self.patch_size = patch_size
        self.img_size = img_size

    def forward(self, x):
        batch_size = x.shape[0]
        features = self.vit_backbone.forward_features(x)

        # Classification output comes from the [CLS] token
        cls_token_features = features[:, 0]
        classification_output = self.classification_head(cls_token_features)

        # Segmentation output
        patch_features = features[:, 1:] # All tokens except the [CLS] token

        num_patches_h = self.img_size[0] // self.patch_size
        num_patches_w = self.img_size[1] // self.patch_size

        # Reshape to (batch_size, embed_dim, num_patches_h, num_patches_w) for ConvTranspose2d
        spatial_features = patch_features.permute(0, 2, 1).view(batch_size, self.vit_backbone.num_features, num_patches_h, num_patches_w)

        # Pass through the U-Net style decoder
        x = self.up1(spatial_features)
        x = self.up2(x)
        x = self.up3(x)
        x = self.up4(x) # Added new upsampling block
        segmentation_output = self.final_seg_conv(x)

        return classification_output, segmentation_output

# Clear CUDA cache and collect garbage before model instantiation and transfer to device
torch.cuda.empty_cache()
gc.collect()

# Instantiate the model and move to device (updated patch_size parameter)
model = DiTMultiTask(num_classes=num_classes, segmentation_output_channels=1, patch_size=16).to(device) # Reverted patch_size here as well

print("DiT-MultiTask model initialized.")
# print(model) # Keep this commented to avoid verbose output unless needed

# Define loss functions
classification_criterion = nn.CrossEntropyLoss()

# Initialize individual segmentation loss functions
tversky_loss_fn = TverskyLoss(alpha=0.3, beta=0.7)
focal_loss_fn = FocalLoss(alpha=0.25, gamma=2.0)
boundary_loss_fn = BoundaryLoss(scale_factor=10.0)

# Initialize the combined segmentation loss
segmentation_criterion = CombinedSegmentationLoss(
    tversky_loss_fn=tversky_loss_fn,
    focal_loss_fn=focal_loss_fn,
    boundary_loss_fn=boundary_loss_fn,
    tversky_weight=1.0, # Adjust weights as needed during experimentation
    focal_weight=1.0,
    boundary_weight=1.0
)

# Define optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

print("Loss functions and optimizer initialized.")


# Training function
def train_model(model, train_loader, val_loader, classification_criterion, segmentation_criterion, optimizer, device, num_epochs=10, gradient_accumulation_steps=4):
    model.to(device)
    scaler = torch.amp.GradScaler('cuda') # Initialize GradScaler for mixed precision training

    # Store metrics
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'val_dice': []} # Added val_dice

    print("Starting training...")
    for epoch in range(num_epochs):
        model.train() # Set model to training mode
        running_loss = 0.0
        correct_predictions = 0
        total_predictions = 0

        # Reset gradients at the beginning of each accumulation cycle, not each batch
        optimizer.zero_grad()

        for i, (inputs, masks, labels) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} (Train)")):
            inputs = inputs.to(device)
            masks = masks.to(device) # Move masks to device
            labels = labels.to(device)

            # Ensure masks are float32 for loss calculation
            masks = masks.float()

            # Forward pass with autocast for mixed precision
            with torch.amp.autocast(device_type='cuda'): # Updated autocast call with device_type
                classification_outputs, segmentation_outputs = model(inputs)

                # Calculate classification loss
                loss_cls = classification_criterion(classification_outputs, labels)

                # Calculate segmentation loss using the combined criterion
                loss_seg = segmentation_criterion(segmentation_outputs, masks)

                # Combined loss (you can adjust weights for each task)
                total_loss = loss_cls + 1.0 * loss_seg # Increased segmentation loss weight

                # Normalize the loss by the number of accumulation steps
                # This ensures the effective learning rate remains consistent
                total_loss = total_loss / gradient_accumulation_steps

            # Backward pass and scale loss for mixed precision
            scaler.scale(total_loss).backward()

            # Perform optimizer step and update scaler only after accumulating gradients
            if (i + 1) % gradient_accumulation_steps == 0 or (i + 1) == len(train_loader):
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad() # Clear gradients for the next accumulation cycle

            # Note: running_loss accumulates the scaled loss, so it reflects the average loss per step, not per sample
            running_loss += total_loss.item() * inputs.size(0) * gradient_accumulation_steps # Scale back to get original loss for metrics

            # Calculate training accuracy for classification
            _, predicted = torch.max(classification_outputs.data, 1)
            total_predictions += labels.size(0)
            correct_predictions += (predicted == labels).sum().item()

        epoch_train_loss = running_loss / len(train_loader.dataset)
        epoch_train_acc = correct_predictions / total_predictions
        history['train_loss'].append(epoch_train_loss)
        history['train_acc'].append(epoch_train_acc)

        # Validation phase
        model.eval() # Set model to evaluation mode
        val_running_loss = 0.0
        val_correct_predictions = 0
        val_total_predictions = 0
        val_running_dice = 0.0 # Initialize for DSC

        with torch.no_grad(): # No gradient calculation during validation
            for inputs, masks, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} (Validation)"):
                inputs = inputs.to(device)
                masks = masks.to(device) # Move masks to device
                labels = labels.to(device)

                # Ensure masks are float32 for loss calculation
                masks = masks.float()

                with torch.amp.autocast(device_type='cuda'): # Updated autocast call with device_type
                    classification_outputs, segmentation_outputs = model(inputs)

                    loss_cls = classification_criterion(classification_outputs, labels)
                    # Change: Use combined segmentation loss
                    loss_seg = segmentation_criterion(segmentation_outputs, masks)
                    val_loss = loss_cls + 1.0 * loss_seg # Keep consistent weighting

                val_running_loss += val_loss.item() * inputs.size(0)

                _, predicted = torch.max(classification_outputs.data, 1)
                val_total_predictions += labels.size(0)
                val_correct_predictions += (predicted == labels).sum().item()

                # Calculate DSC for segmentation
                batch_dice_score = dice_score(segmentation_outputs, masks)
                val_running_dice += batch_dice_score * inputs.size(0) # Accumulate total dice score weighted by batch size

        epoch_val_loss = val_running_loss / len(val_loader.dataset)
        epoch_val_acc = val_correct_predictions / val_total_predictions
        epoch_val_dice = val_running_dice / len(val_loader.dataset) # Average DSC for the epoch

        history['val_loss'].append(epoch_val_loss)
        history['val_acc'].append(epoch_val_acc)
        history['val_dice'].append(epoch_val_dice) # Store epoch DSC

        print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {epoch_train_loss:.4f}, Train Acc: {epoch_train_acc:.4f} | Val Loss: {epoch_val_loss:.4f}, Val Acc: {epoch_val_acc:.4f} | Val DSC: {epoch_val_dice:.4f}")

    print("Training complete!")
    return history

# Run the training
num_epochs = 5 # You can adjust the number of epochs
gradient_accumulation_steps = 4 # Define gradient accumulation steps
training_history = train_model(model, train_loader, val_loader, classification_criterion, segmentation_criterion, optimizer, device, num_epochs=num_epochs, gradient_accumulation_steps=gradient_accumulation_steps)


### Classification Results (3x5 Grid)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import random
import torch.amp # Import for Automatic Mixed Precision

# Get class names from the dataset
idx_to_class = {v: k for k, v in dataset.class_to_idx.items()}

model.eval() # Set model to evaluation mode

num_classification_samples = 15 # For a 3x5 grid
sample_indices_cls = random.sample(range(len(val_dataset)), num_classification_samples)

fig_cls, axes_cls = plt.subplots(3, 5, figsize=(18, 12))
fig_cls.suptitle('Classification Predictions', fontsize=16)

for i, idx in enumerate(sample_indices_cls):
    row = i // 5
    col = i % 5

    # Correctly unpack the three values returned by val_dataset[idx]
    original_image, _, true_label_idx = val_dataset[idx]

    # Convert tensor to image for display (denormalize)
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    display_image = original_image.cpu().numpy().transpose(1, 2, 0) # C, H, W -> H, W, C
    display_image = std * display_image + mean # Denormalize
    display_image = np.clip(display_image, 0, 1)

    true_label = idx_to_class[true_label_idx]

    # Add batch dimension and move to device for model inference
    input_tensor = original_image.unsqueeze(0).to(device) # No .half() needed for inference here since model is FP32

    with torch.no_grad():
        with torch.amp.autocast(device_type='cuda'): # Use autocast for mixed precision inference
            cls_output, _ = model(input_tensor)

    # Get predicted classification label
    _, predicted_label_idx = torch.max(cls_output.data, 1)
    predicted_label = idx_to_class[predicted_label_idx.item()]

    # Determine color based on prediction correctness
    title_color = 'green' if predicted_label == true_label else 'red'

    axes_cls[row, col].imshow(display_image)
    axes_cls[row, col].set_title(f"True:  {true_label}\nPred: {predicted_label}", fontsize=10, color=title_color)
    axes_cls[row, col].axis('off')

plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout to prevent suptitle overlap
plt.show()

### Segmentation Results (3-Row Format)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import random
import torch.amp # Import for Automatic Mixed Precision
from torchvision.utils import make_grid

# Get class names from the dataset
idx_to_class = {v: k for k, v in dataset.class_to_idx.items()}

model.eval() # Set model to evaluation mode

num_segmentation_samples = 5 # Display 5 samples as requested
sample_indices_seg = random.sample(range(len(val_dataset)), num_segmentation_samples)

fig_seg, axes_seg = plt.subplots(num_segmentation_samples, 3, figsize=(15, num_segmentation_samples * 5))
fig_seg.suptitle('Segmentation Predictions', fontsize=16)

for i, idx in enumerate(sample_indices_seg):
    original_image_tensor, true_mask_tensor, true_label_idx = val_dataset[idx]

    # Denormalize image for display
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    display_image = original_image_tensor.cpu().numpy().transpose(1, 2, 0) # C, H, W -> H, W, C
    display_image = std * display_image + mean # Denormalize
    display_image = np.clip(display_image, 0, 1) # Clip to ensure valid pixel range

    true_label = idx_to_class[true_label_idx]

    # Add batch dimension and move to device for model inference
    input_tensor = original_image_tensor.unsqueeze(0).to(device) # No .half() needed for inference here since model is FP32

    with torch.no_grad():
        with torch.amp.autocast(device_type='cuda'): # Use autocast for mixed precision inference
            _, seg_output = model(input_tensor)

    # Process predicted segmentation mask
    predicted_mask = torch.sigmoid(seg_output).squeeze().cpu().numpy() # Apply sigmoid, remove batch/channel, convert to numpy

    # Plot Original Image
    axes_seg[i, 0].imshow(display_image)
    axes_seg[i, 0].set_title(f"Original (True: {true_label})")
    axes_seg[i, 0].axis('off')

    # Plot Predicted Mask
    # Using 'gray' colormap for binary-like masks, can be changed if multi-class segmentation
    axes_seg[i, 1].imshow(predicted_mask, cmap='gray')
    axes_seg[i, 1].set_title("Predicted Mask")
    axes_seg[i, 1].axis('off')

    # Plot Mask Overlay
    axes_seg[i, 2].imshow(display_image, cmap='gray')
    axes_seg[i, 2].imshow(predicted_mask, cmap='jet', alpha=0.5) # Overlay with transparency
    axes_seg[i, 2].set_title("Mask Overlay")
    axes_seg[i, 2].axis('off')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

### Classification Confusion Matrix

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import torch

model.eval() # Set model to evaluation mode

all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, masks, labels in val_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        with torch.amp.autocast(device_type='cuda'): # Use autocast for mixed precision inference
            classification_outputs, _ = model(inputs)

        _, predicted = torch.max(classification_outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Compute confusion matrix
cm = confusion_matrix(all_labels, all_preds)

# Get class names
class_names = [idx_to_class[i] for i in sorted(idx_to_class.keys())]

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix for Classification')
plt.show()

## Implement Tversky Loss

### Subtask:
Define a custom PyTorch module or function for Tversky Loss with hyperparameters alpha=0.3 and beta=0.7. This loss is effective for imbalanced datasets common in segmentation.


**Reasoning**:
Implementing the `TverskyLoss` as a PyTorch `nn.Module` allows it to be easily integrated into the training loop and benefit from PyTorch's automatic differentiation. I will include the `torch.sigmoid` activation within the forward method and handle the calculation of true positives, false positives, and false negatives, along with a small epsilon for numerical stability.



In [ ]:
import torch
import torch.nn as nn

class TverskyLoss(nn.Module):
    def __init__(self, alpha: float = 0.3, beta: float = 0.7, epsilon: float = 1e-6):
        super(TverskyLoss, self).__init__()
        self.alpha = alpha
        self.beta = beta
        self.epsilon = epsilon

    def forward(self, y_pred: torch.Tensor, y_true: torch.Tensor) -> torch.Tensor:
        # Apply sigmoid to predictions to get probabilities
        y_pred = torch.sigmoid(y_pred)

        # Flatten tensors for easier calculation
        y_pred = y_pred.view(-1)
        y_true = y_true.view(-1)

        # Calculate True Positives, False Positives, False Negatives
        true_positives = (y_pred * y_true).sum()
        false_positives = ((1 - y_true) * y_pred).sum()
        false_negatives = (y_true * (1 - y_pred)).sum()

        # Calculate Tversky Index
        # Formula: (TP + epsilon) / (TP + alpha * FN + beta * FP + epsilon)
        numerator = true_positives + self.epsilon
        denominator = true_positives + self.alpha * false_negatives + self.beta * false_positives + self.epsilon

        tversky_index = numerator / denominator
        tversky_loss = 1.0 - tversky_index

        return tversky_loss

print("TverskyLoss class defined with default alpha=0.3 and beta=0.7.")

## Implement Focal Loss

### Subtask:
Define a custom PyTorch module or function for Focal Loss. This loss addresses the class imbalance by down-weighting the loss assigned to well-classified examples.


**Reasoning**:
Implementing the `FocalLoss` as a PyTorch `nn.Module` allows it to be easily integrated into the training loop and benefit from PyTorch's automatic differentiation. This implementation will handle `gamma` and `alpha` parameters and use sigmoid on predictions.



In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class FocalLoss(nn.Module):
    def __init__(self, alpha: float = 0.25, gamma: float = 2.0, epsilon: float = 1e-6):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon

    def forward(self, y_pred_logits: torch.Tensor, y_true: torch.Tensor) -> torch.Tensor:
        # Flatten tensors for easier calculation
        y_pred_logits_flat = y_pred_logits.view(-1)
        y_true_flat = y_true.view(-1)

        # Calculate probabilities from logits for pt calculation
        y_pred_proba = torch.sigmoid(y_pred_logits_flat)

        # Calculate pt
        pt = y_pred_proba * y_true_flat + (1 - y_pred_proba) * (1 - y_true_flat)

        # Clip pt to avoid log(0) issues
        pt = torch.clamp(pt, self.epsilon, 1. - self.epsilon)

        # Calculate binary cross entropy with logits for each pixel
        # This is the crucial change to make it autocast-safe
        bce = F.binary_cross_entropy_with_logits(y_pred_logits_flat, y_true_flat, reduction='none')

        # Apply alpha weighting
        alpha_t = self.alpha * y_true_flat + (1 - self.alpha) * (1 - y_true_flat)

        focal_loss = -alpha_t * ((1 - pt) ** self.gamma) * bce

        return focal_loss.mean()

print("FocalLoss class defined with default alpha=0.25 and gamma=2.0.")

# Task
Implement and integrate advanced segmentation loss functions (Tversky Loss, Focal Loss, and Boundary Loss) into an existing multi-task Vision Transformer (ViT) model for medical image analysis. Subsequently, update the training loop to utilize these new loss functions and observe their impact on segmentation performance.

## Implement Boundary Loss

### Subtask:
Define a custom PyTorch module or function for a simplified Boundary Loss. This loss focuses on the region around the segmentation boundary, which helps to refine the edges of predicted masks. A common approach involves using a distance map.


**Reasoning**:
Implementing the `BoundaryLoss` as a PyTorch `nn.Module` allows for easy integration into the training loop. This implementation will compute a weighted Binary Cross-Entropy loss, where the weights are derived from the absolute signed distance map (SDM) of the ground truth mask. Pixels closer to the true boundary will receive higher weights, thus focusing the loss on refining segmentation edges. The SDM calculation will leverage `scipy.ndimage.distance_transform_edt` and will be performed per batch item.



In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from scipy.ndimage import distance_transform_edt

class BoundaryLoss(nn.Module):
    def __init__(self, scale_factor: float = 10.0):
        super(BoundaryLoss, self).__init__()
        # scale_factor controls the decay rate of the exponential weight
        # A smaller scale_factor means the weight decays faster, focusing more sharply on the boundary.
        # A larger scale_factor means the weight decays slower, affecting a wider region around the boundary.
        self.scale_factor = scale_factor

    def forward(self, y_pred_logits: torch.Tensor, y_true: torch.Tensor) -> torch.Tensor:
        # Ensure y_true is binary (0 or 1) and of float type for calculations
        y_true_float = y_true.float()

        # Get the device (CPU/GPU) where y_pred_logits are located for new tensor creation
        device = y_pred_logits.device

        # Initialize list to store absolute SDMs for the batch
        batch_abs_sdms = []

        # Iterate over each item in the batch to compute the Signed Distance Map (SDM)
        # This part uses numpy and scipy on CPU, which is acceptable since y_true is a fixed target
        for i in range(y_true_float.shape[0]):
            # Convert single-channel mask tensor to numpy array for scipy
            # Assuming y_true has shape (Batch, Channels, Height, Width) and Channels=1
            mask_np = y_true_float[i, 0].cpu().numpy()

            # Compute distance transform to nearest background (0) pixel for foreground (1) pixels
            dist_to_background = distance_transform_edt(mask_np)
            # Compute distance transform to nearest foreground (1) pixel for background (0) pixels
            dist_to_foreground = distance_transform_edt(1 - mask_np)

            # Create Signed Distance Map (SDM):
            # Positive inside the object, negative outside, and zero at the boundary
            sdm_np = dist_to_background - dist_to_foreground

            # Take the absolute value to get the unsigned distance to the boundary
            # Pixels closer to the boundary will have smaller abs_sdm values
            abs_sdm_np = np.abs(sdm_np)

            # Convert the numpy array back to a PyTorch tensor and move it to the correct device
            # Add a channel dimension back to match the expected shape for element-wise ops later
            batch_abs_sdms.append(torch.from_numpy(abs_sdm_np).float().unsqueeze(0).to(device))

        # Stack the list of absolute SDM tensors to form a single batch tensor
        abs_sdm_true = torch.stack(batch_abs_sdms, dim=0)

        # Remove the channel dimension from predictions, target, and SDM if it's singleton (e.g., (B,1,H,W) -> (B,H,W))
        y_pred_logits = y_pred_logits.squeeze(1)
        y_true_float = y_true_float.squeeze(1)
        abs_sdm_true = abs_sdm_true.squeeze(1)

        # Create a weight map that gives higher weights to pixels closer to the boundary.
        # The exponential decay ensures that weights are highest at the boundary (distance = 0) and decrease further away.
        # Adding 1.0 ensures that all pixels contribute to the loss, with boundary regions being emphasized.
        weight_map = 1.0 + torch.exp(-abs_sdm_true / self.scale_factor)

        # Calculate the Binary Cross-Entropy with Logits loss for each pixel
        bce_per_pixel = F.binary_cross_entropy_with_logits(y_pred_logits, y_true_float, reduction='none')

        # Apply the computed weight map to the per-pixel BCE loss
        weighted_bce_loss = bce_per_pixel * weight_map

        # Take the mean of the weighted loss over all pixels and batch items to get the final Boundary Loss
        boundary_loss = weighted_bce_loss.mean()

        return boundary_loss

print("BoundaryLoss class defined using scipy.ndimage for signed distance map computation.")

## Combine Advanced Segmentation Losses

### Subtask:
Create a new combined segmentation loss function that integrates the Tversky Loss, Focal Loss, and Boundary Loss. Each component will be weighted appropriately (e.g., sum of losses, or with adjustable coefficients) to create a robust segmentation criterion.


**Reasoning**:
The first step is to define the `CombinedSegmentationLoss` class and its `__init__` method, where the individual loss functions and their respective weights will be initialized. Then, define the `forward` method to compute and sum the weighted losses.



In [ ]:
import torch
import torch.nn as nn

# Assuming TverskyLoss, FocalLoss, and BoundaryLoss are already defined in the environment

class CombinedSegmentationLoss(nn.Module):
    def __init__(self,
                 tversky_loss_fn: TverskyLoss,
                 focal_loss_fn: FocalLoss,
                 boundary_loss_fn: BoundaryLoss,
                 tversky_weight: float = 1.0,
                 focal_weight: float = 1.0,
                 boundary_weight: float = 1.0):
        super(CombinedSegmentationLoss, self).__init__()
        self.tversky_loss_fn = tversky_loss_fn
        self.focal_loss_fn = focal_loss_fn
        self.boundary_loss_fn = boundary_loss_fn
        self.tversky_weight = tversky_weight
        self.focal_weight = focal_weight
        self.boundary_weight = boundary_weight

    def forward(self, y_pred_logits: torch.Tensor, y_true: torch.Tensor) -> torch.Tensor:
        # Ensure y_true is the correct type (float) for loss calculations
        y_true_float = y_true.float()

        # Calculate individual losses
        loss_tversky = self.tversky_loss_fn(y_pred_logits, y_true_float)
        loss_focal = self.focal_loss_fn(y_pred_logits, y_true_float)
        loss_boundary = self.boundary_loss_fn(y_pred_logits, y_true_float)

        # Combine losses with weights
        combined_loss = (
            self.tversky_weight * loss_tversky +
            self.focal_weight * loss_focal +
            self.boundary_weight * loss_boundary
        )

        return combined_loss

print("CombinedSegmentationLoss class defined, integrating Tversky, Focal, and Boundary losses.")

## Update Training Loop with New Loss

### Subtask:
Modify the `train_model` function to replace `BCEWithLogitsLoss` with the newly defined combined segmentation loss. Ensure that the loss is correctly applied during both training and validation phases, and that the segmentation output is handled appropriately (e.g., sigmoid activation before loss calculation if needed).


**Reasoning**:
Re-run the cell containing the model definition, loss function initialization, and training loop, as the `FocalLoss` has now been corrected for `autocast` compatibility.



In [ ]:
import torch.nn as nn
import timm # PyTorch Image Models library for ViT
import torch
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, random_split
from PIL import Image
import os
import gc # Import garbage collector
import torch.amp # Import for Automatic Mixed Precision
from tqdm import tqdm # Import tqdm for progress bars

# Check if CUDA is available and set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Define the custom dataset for classification AND segmentation
class BrainTumorDataset(Dataset):
    def __init__(self, root_dir, mask_root_dir, transform=None, mask_transform=None):
        self.root_dir = root_dir
        self.mask_root_dir = mask_root_dir
        self.transform = transform
        self.mask_transform = mask_transform
        self.image_paths = []
        self.mask_paths = [] # Store mask paths
        self.labels = []
        self.class_to_idx = {}

        # Collect all image paths, corresponding mask paths, and assign labels
        for i, class_name in enumerate(sorted(os.listdir(root_dir))):
            class_path = os.path.join(root_dir, class_name)
            if os.path.isdir(class_path):
                self.class_to_idx[class_name] = i
                for img_name in os.listdir(class_path):
                    if img_name.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp')):
                        image_full_path = os.path.join(class_path, img_name)

                        # Correctly construct mask path:
                        # Take the base name of the image and append .png extension for the mask
                        base_img_name = os.path.splitext(img_name)[0]
                        mask_name = base_img_name + '.png'
                        mask_full_path = os.path.join(mask_root_dir, mask_name)

                        if os.path.exists(mask_full_path):
                            self.image_paths.append(image_full_path)
                            self.mask_paths.append(mask_full_path)
                            self.labels.append(i)
                        else:
                            print(f"Warning: No mask found for image {img_name} at {mask_full_path}. Skipping.")

        print(f"Found {len(self.image_paths)} images and masks belonging to {len(self.class_to_idx)} classes: {self.class_to_idx}")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        mask_path = self.mask_paths[idx]
        image = Image.open(img_path).convert('RGB')
        mask = Image.open(mask_path).convert('L') # Load mask as grayscale
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)
        if self.mask_transform:
            mask = self.mask_transform(mask)

        return image, mask, label # Return image, mask, and label

# Define transformations
# Image transforms: Resize, ToTensor, Normalize
image_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Mask transforms: Resize, ToTensor (masks are typically 0 or 1, no normalization)
# Masks should be loaded as grayscale and converted to float tensor.
mask_transform = transforms.Compose([
    transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.NEAREST), # Use NEAREST for masks
    transforms.ToTensor() # Converts to [0,1] float automatically
])


# NOTE: `path` variable (from kagglehub.dataset_download) is available from cell 'vzahEgYWIe4H'.
# Re-evaluate base paths for robustness if this cell is run independently.
if 'path' not in locals():
    import kagglehub
    path = kagglehub.dataset_download("briscdataset/brisc2025")
    print("kagglehub dataset downloaded to ensure 'path' is defined.")

base_train_image_path = os.path.join(path, 'brisc2025', 'classification_task', 'train')
base_train_mask_path = os.path.join(path, 'brisc2025', 'segmentation_task', 'train', 'masks') # New mask path

# Instantiate the dataset
dataset = BrainTumorDataset(
    root_dir=base_train_image_path,
    mask_root_dir=base_train_mask_path,
    transform=image_transform,
    mask_transform=mask_transform
)

# Split the dataset into training and validation sets
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

print(f"Training set size: {len(train_dataset)}")
print(f"Validation set size: {len(val_dataset)}")

# Create DataLoaders
batch_size = 4 # Reduced further to free up GPU memory
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

print("DataLoaders created successfully.")

# Number of output classes for classification
num_classes = len(dataset.class_to_idx)
print(f"Number of classification classes: {num_classes}")

# Define a U-Net style UpBlock for the decoder
class UpBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.up = nn.Sequential(
            nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2), # Upsample by 2
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.up(x)

class DiTMultiTask(nn.Module):
    def __init__(self, num_classes, segmentation_output_channels=1, img_size=(224, 224), patch_size=16):
        super().__init__()
        # Using an even SMALLER pre-trained Vision Transformer (ViT) as the backbone to save memory
        # Now using patch_size=16 for the vit_tiny model
        self.vit_backbone = timm.create_model(f'vit_tiny_patch{patch_size}_224', pretrained=True, num_classes=0)

        # Dynamically get embed_dim from the backbone
        embed_dim = self.vit_backbone.num_features # For vit_tiny, this is typically 192

        # Classification Head
        self.classification_head = nn.Linear(embed_dim, num_classes)

        # Segmentation Head (U-Net style decoder for progressive upsampling)
        # Input to decoder will be (batch, embed_dim, 224/patch_size, 224/patch_size)
        # For patch_size=16, this is (batch, 192, 14, 14)

        # Define output dimensions for each upsampling stage
        decoder_dims = [embed_dim, embed_dim // 2, embed_dim // 4]

        # Adjust upsampling layers for patch_size=16 input (14x14 spatial resolution)
        self.up1 = UpBlock(decoder_dims[0], decoder_dims[1]) # (192, 14, 14) -> (96, 28, 28)
        self.up2 = UpBlock(decoder_dims[1], decoder_dims[2]) # (96, 28, 28) -> (48, 56, 56)
        self.up3 = UpBlock(decoder_dims[2], decoder_dims[2] // 2) # (48, 56, 56) -> (24, 112, 112)
        # Need one more upsampling block to reach 224x224
        self.up4 = UpBlock(decoder_dims[2] // 2, decoder_dims[2] // 4) # (24, 112, 112) -> (12, 224, 224)

        self.final_seg_conv = nn.Conv2d(decoder_dims[2] // 4, segmentation_output_channels, kernel_size=1)

        self.patch_size = patch_size
        self.img_size = img_size

    def forward(self, x):
        batch_size = x.shape[0]
        features = self.vit_backbone.forward_features(x)

        # Classification output comes from the [CLS] token
        cls_token_features = features[:, 0]
        classification_output = self.classification_head(cls_token_features)

        # Segmentation output
        patch_features = features[:, 1:] # All tokens except the [CLS] token

        num_patches_h = self.img_size[0] // self.patch_size
        num_patches_w = self.img_size[1] // self.patch_size

        # Reshape to (batch_size, embed_dim, num_patches_h, num_patches_w) for ConvTranspose2d
        spatial_features = patch_features.permute(0, 2, 1).view(batch_size, self.vit_backbone.num_features, num_patches_h, num_patches_w)

        # Pass through the U-Net style decoder
        x = self.up1(spatial_features)
        x = self.up2(x)
        x = self.up3(x)
        x = self.up4(x) # Added new upsampling block
        segmentation_output = self.final_seg_conv(x)

        return classification_output, segmentation_output

# Clear CUDA cache and collect garbage before model instantiation and transfer to device
torch.cuda.empty_cache()
gc.collect()

# Instantiate the model and move to device (updated patch_size parameter)
model = DiTMultiTask(num_classes=num_classes, segmentation_output_channels=1, patch_size=16).to(device) # Reverted patch_size here as well

print("DiT-MultiTask model initialized.")
# print(model) # Keep this commented to avoid verbose output unless needed

# Define loss functions
classification_criterion = nn.CrossEntropyLoss()

# Initialize individual segmentation loss functions
tversky_loss_fn = TverskyLoss(alpha=0.3, beta=0.7)
focal_loss_fn = FocalLoss(alpha=0.25, gamma=2.0)
boundary_loss_fn = BoundaryLoss(scale_factor=10.0)

# Initialize the combined segmentation loss
segmentation_criterion = CombinedSegmentationLoss(
    tversky_loss_fn=tversky_loss_fn,
    focal_loss_fn=focal_loss_fn,
    boundary_loss_fn=boundary_loss_fn,
    tversky_weight=1.0, # Adjust weights as needed during experimentation
    focal_weight=1.0,
    boundary_weight=1.0
)

# Define optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

print("Loss functions and optimizer initialized.")


# Training function
def train_model(model, train_loader, val_loader, classification_criterion, segmentation_criterion, optimizer, device, num_epochs=10, gradient_accumulation_steps=4):
    model.to(device)
    scaler = torch.amp.GradScaler('cuda') # Initialize GradScaler for mixed precision training

    # Store metrics
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    print("Starting training...")
    for epoch in range(num_epochs):
        model.train() # Set model to training mode
        running_loss = 0.0
        correct_predictions = 0
        total_predictions = 0

        # Reset gradients at the beginning of each accumulation cycle, not each batch
        optimizer.zero_grad()

        for i, (inputs, masks, labels) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} (Train)")):
            inputs = inputs.to(device)
            masks = masks.to(device) # Move masks to device
            labels = labels.to(device)

            # Ensure masks are float32 for loss calculation
            masks = masks.float()

            # Forward pass with autocast for mixed precision
            with torch.amp.autocast(device_type='cuda'): # Updated autocast call with device_type
                classification_outputs, segmentation_outputs = model(inputs)

                # Calculate classification loss
                loss_cls = classification_criterion(classification_outputs, labels)

                # Calculate segmentation loss using the combined criterion
                loss_seg = segmentation_criterion(segmentation_outputs, masks)

                # Combined loss (you can adjust weights for each task)
                total_loss = loss_cls + 0.5 * loss_seg # Increased segmentation loss weight

                # Normalize the loss by the number of accumulation steps
                # This ensures the effective learning rate remains consistent
                total_loss = total_loss / gradient_accumulation_steps

            # Backward pass and scale loss for mixed precision
            scaler.scale(total_loss).backward()

            # Perform optimizer step and update scaler only after accumulating gradients
            if (i + 1) % gradient_accumulation_steps == 0 or (i + 1) == len(train_loader):
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad() # Clear gradients for the next accumulation cycle

            # Note: running_loss accumulates the scaled loss, so it reflects the average loss per step, not per sample
            running_loss += total_loss.item() * inputs.size(0) * gradient_accumulation_steps # Scale back to get original loss for metrics

            # Calculate training accuracy for classification
            _, predicted = torch.max(classification_outputs.data, 1)
            total_predictions += labels.size(0)
            correct_predictions += (predicted == labels).sum().item()

        epoch_train_loss = running_loss / len(train_loader.dataset)
        epoch_train_acc = correct_predictions / total_predictions
        history['train_loss'].append(epoch_train_loss)
        history['train_acc'].append(epoch_train_acc)

        # Validation phase
        model.eval() # Set model to evaluation mode
        val_running_loss = 0.0
        val_correct_predictions = 0
        val_total_predictions = 0

        with torch.no_grad(): # No gradient calculation during validation
            for inputs, masks, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} (Validation)"):
                inputs = inputs.to(device)
                masks = masks.to(device) # Move masks to device
                labels = labels.to(device)

                # Ensure masks are float32 for loss calculation
                masks = masks.float()

                with torch.amp.autocast(device_type='cuda'): # Updated autocast call with device_type
                    classification_outputs, segmentation_outputs = model(inputs)

                    loss_cls = classification_criterion(classification_outputs, labels)
                    # Change: Use combined segmentation loss
                    loss_seg = segmentation_criterion(segmentation_outputs, masks)
                    val_loss = loss_cls + 0.5 * loss_seg # Keep consistent weighting

                val_running_loss += val_loss.item() * inputs.size(0)

                _, predicted = torch.max(classification_outputs.data, 1)
                val_total_predictions += labels.size(0)
                val_correct_predictions += (predicted == labels).sum().item()

        epoch_val_loss = val_running_loss / len(val_loader.dataset)
        epoch_val_acc = val_correct_predictions / val_total_predictions # Define epoch_val_acc
        history['val_loss'].append(epoch_val_loss)
        history['val_acc'].append(epoch_val_acc)

        print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {epoch_train_loss:.4f}, Train Acc: {epoch_train_acc:.4f} | Val Loss: {epoch_val_loss:.4f}, Val Acc: {epoch_val_acc:.4f}")

    print("Training complete!")
    return history

# Run the training
num_epochs = 5 # You can adjust the number of epochs
gradient_accumulation_steps = 4 # Define gradient accumulation steps
training_history = train_model(model, train_loader, val_loader, classification_criterion, segmentation_criterion, optimizer, device, num_epochs=num_epochs, gradient_accumulation_steps=gradient_accumulation_steps)

**Reasoning**:
The `CombinedSegmentationLoss` class needs to be re-defined to incorporate the corrected `FocalLoss` class. I will re-run the cell that defines `CombinedSegmentationLoss`.



In [ ]:
import torch
import torch.nn as nn

# Assuming TverskyLoss, FocalLoss, and BoundaryLoss are already defined in the environment

class CombinedSegmentationLoss(nn.Module):
    def __init__(self,
                 tversky_loss_fn: TverskyLoss,
                 focal_loss_fn: FocalLoss,
                 boundary_loss_fn: BoundaryLoss,
                 tversky_weight: float = 1.0,
                 focal_weight: float = 1.0,
                 boundary_weight: float = 1.0):
        super(CombinedSegmentationLoss, self).__init__()
        self.tversky_loss_fn = tversky_loss_fn
        self.focal_loss_fn = focal_loss_fn
        self.boundary_loss_fn = boundary_loss_fn
        self.tversky_weight = tversky_weight
        self.focal_weight = focal_weight
        self.boundary_weight = boundary_weight

    def forward(self, y_pred_logits: torch.Tensor, y_true: torch.Tensor) -> torch.Tensor:
        # Ensure y_true is the correct type (float) for loss calculations
        y_true_float = y_true.float()

        # Calculate individual losses
        loss_tversky = self.tversky_loss_fn(y_pred_logits, y_true_float)
        loss_focal = self.focal_loss_fn(y_pred_logits, y_true_float)
        loss_boundary = self.boundary_loss_fn(y_pred_logits, y_true_float)

        # Combine losses with weights
        combined_loss = (
            self.tversky_weight * loss_tversky +
            self.focal_weight * loss_focal +
            self.boundary_weight * loss_boundary
        )

        return combined_loss

print("CombinedSegmentationLoss class defined, integrating Tversky, Focal, and Boundary losses.")

## Implement Tversky Loss

### Subtask:
Define a custom PyTorch module or function for Tversky Loss with hyperparameters alpha=0.3 and beta=0.7. This loss is effective for imbalanced datasets common in segmentation.


## Final Task

### Subtask:
Confirm the successful integration and application of the Tversky, Focal, and Boundary losses in the training process, and prepare to re-run the training to observe the impact on segmentation performance.


# Task
Implement and integrate advanced segmentation loss functions (Tversky Loss, Focal Loss, and Boundary Loss) into an existing multi-task Vision Transformer (ViT) model for medical image analysis. Subsequently, update the training loop to utilize these new loss functions and observe their impact on segmentation performance. The plan involves visualizing segmentation predictions and then summarizing the observations.

## Re-run Training with Corrected Loss Functions

### Subtask:
Re-run the training process with the updated combined segmentation loss functions (Tversky, Focal, and Boundary Loss) to ensure the model trains successfully.


**Reasoning**:
The model has finished training, and the next step is to visualize the segmentation results to evaluate the impact of the new loss functions. This involves generating and displaying images with predicted masks and overlays.



In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import random
import torch.amp # Import for Automatic Mixed Precision
from torchvision.utils import make_grid

# Get class names from the dataset
idx_to_class = {v: k for k, v in dataset.class_to_idx.items()}

model.eval() # Set model to evaluation mode

num_segmentation_samples = 5 # Display 5 samples as requested
sample_indices_seg = random.sample(range(len(val_dataset)), num_segmentation_samples)

fig_seg, axes_seg = plt.subplots(num_segmentation_samples, 3, figsize=(15, num_segmentation_samples * 5))
fig_seg.suptitle('Segmentation Predictions', fontsize=16)

for i, idx in enumerate(sample_indices_seg):
    original_image_tensor, true_mask_tensor, true_label_idx = val_dataset[idx]

    # Denormalize image for display
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    display_image = original_image_tensor.cpu().numpy().transpose(1, 2, 0) # C, H, W -> H, W, C
    display_image = std * display_image + mean # Denormalize
    display_image = np.clip(display_image, 0, 1) # Clip to ensure valid pixel range

    true_label = idx_to_class[true_label_idx]

    # Add batch dimension and move to device for model inference
    input_tensor = original_image_tensor.unsqueeze(0).to(device) # No .half() needed for inference here since model is FP32

    with torch.no_grad():
        with torch.amp.autocast(device_type='cuda'): # Use autocast for mixed precision inference
            _, seg_output = model(input_tensor)

    # Process predicted segmentation mask
    predicted_mask = torch.sigmoid(seg_output).squeeze().cpu().numpy() # Apply sigmoid, remove batch/channel, convert to numpy

    # Plot Original Image
    axes_seg[i, 0].imshow(display_image)
    axes_seg[i, 0].set_title(f"Original (True: {true_label})")
    axes_seg[i, 0].axis('off')

    # Plot Predicted Mask
    # Using 'gray' colormap for binary-like masks, can be changed if multi-class segmentation
    axes_seg[i, 1].imshow(predicted_mask, cmap='gray')
    axes_seg[i, 1].set_title("Predicted Mask")
    axes_seg[i, 1].axis('off')

    # Plot Mask Overlay
    axes_seg[i, 2].imshow(display_image, cmap='gray')
    axes_seg[i, 2].imshow(predicted_mask, cmap='jet', alpha=0.5) # Overlay with transparency
    axes_seg[i, 2].set_title("Mask Overlay")
    axes_seg[i, 2].axis('off')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

## Visualize Segmentation Predictions

### Subtask:
Generate a 3-row visualization for 5 randomly selected images from the validation set.


## Final Task

### Subtask:
Summarize the qualitative observations from the segmentation visualizations, noting any improvements or remaining issues, and consider next steps for further refinement.


# Task
Implement and integrate advanced segmentation loss functions (Tversky Loss, Focal Loss, and Boundary Loss) into an existing multi-task Vision Transformer (ViT) model for medical image analysis. Subsequently, update the training loop to utilize these new loss functions and observe their impact on segmentation performance. The plan involves visualizing segmentation predictions and then summarizing the observations.

## Analyze Qualitative Segmentation

### Subtask:
Examine the generated segmentation masks and overlays from the validation set. Evaluate the accuracy, completeness, and boundary precision of the predicted masks for different tumor types.


### Qualitative Analysis of Segmentation Predictions

Upon reviewing the generated segmentation masks and overlays, the following observations can be made:

1.  **General Accuracy and Completeness**: The model appears to detect the presence of tumor-like structures in most of the shown examples. The predicted masks generally correspond to regions that visually appear to contain abnormalities in the original MRI scans. However, the completeness of the masks varies; some predictions cover the entire tumor region fairly well (e.g., the meningioma cases), while others seem to only capture a central portion, missing some of the periphery (e.g., some glioma cases).

2.  **Boundary Precision**: The precision of the boundaries in the predicted masks is a significant area for improvement. Many predicted masks exhibit blurry, imprecise, or irregular edges, rather than sharp, well-defined boundaries that accurately delineate the tumor from surrounding healthy tissue. This suggests that while the model identifies the general area of interest, it struggles with fine-grained pixel-level demarcation.

3.  **False Positives/Over-segmentation**: In some instances, the model generates small, disconnected regions in the predicted mask, or appears to slightly over-segment the tumor, extending beyond the clear boundaries seen in the original image. Conversely, there are cases where the prediction is a small, concentrated blob, potentially under-segmenting a larger, more diffuse tumor.

4.  **Performance Across Tumor Types/Views**:
    *   For **meningioma** cases (e.g., the third and fourth rows), the model seems to produce relatively coherent and localized predictions, often capturing the general shape of the tumor. However, even in these cases, boundary precision is not optimal.
    *   For **glioma** cases (e.g., second and fifth rows), the masks tend to be less complete and more diffuse, sometimes appearing as a scattered pattern rather than a solid mass. This might reflect the often infiltrative nature of gliomas, making them harder to delineate, or it could indicate that the model is struggling with the variability in their appearance.
    *   The model also occasionally generates predictions for `no_tumor` or `pituitary` cases, suggesting some false positives or an inability to clearly distinguish between benign structures and tumors without explicit ground truth for those benign structures. However, in the displayed `pituitary` example (first row), the prediction generally corresponds to the pituitary gland region, which could be an acceptable outcome if the task is simply to segment 'abnormal' regions that include this structure, or a false positive if 'pituitary' implies healthy tissue.

5.  **Impact of Loss Functions**: While the model is producing some segmentation, the current visualizations do not immediately suggest a dramatic improvement in boundary quality that might be expected from the Boundary Loss, nor a clear reduction in false positives/negatives due to Focal/Tversky Loss across all samples. This could be due to the limited number of training epochs or the simplified decoder architecture. The losses are integrated, but their full impact might not yet be realized, or their weights might need tuning.

## Identify Improvements and Issues

### Subtask:
Based on the qualitative analysis, identify any noticeable improvements in segmentation performance due to the new loss functions, as well as remaining areas for improvement, such as diffuse boundaries or misclassified regions.


### Improvements and Issues Identified from Qualitative Analysis

**Improvements Due to New Loss Functions:**
*   Based on the current qualitative analysis and the limited number of training epochs, no immediate or dramatic improvements in segmentation performance directly attributable to the new Tversky, Focal, and Boundary loss functions were observed. The model demonstrates a foundational ability to localize tumor-like structures, but the quality of segmentation remains an area for significant development.

**Issues and Areas for Improvement:**
1.  **Boundary Precision**: The most prominent issue is the lack of precise boundaries in predicted masks. Edges are often blurry, irregular, or diffuse, failing to accurately delineate tumor margins from healthy tissue. This suggests that while the Boundary Loss is incorporated, its effect might not be fully realized yet, possibly due to insufficient training or suboptimal weighting.
2.  **Completeness of Masks**: The completeness of predicted masks varies. Some tumors are only partially segmented, with the model often capturing only a central portion while missing peripheral regions. This indicates potential challenges in identifying the full extent of lesions, especially for diffuse tumor types like gliomas.
3.  **False Positives/Over-segmentation**: The model sometimes produces small, extraneous segmented regions or tends to over-segment, extending beyond the visible tumor boundaries. Conversely, under-segmentation for larger masses was also noted.
4.  **Performance Across Tumor Types**: While meningiomas tend to be better localized, gliomas often result in scattered or incomplete masks, reflecting the difficulty in segmenting their infiltrative nature. Also, `no_tumor` and `pituitary` cases occasionally show false positive segmentations.
5.  **Limited Impact of Loss Functions (Initial Observation)**: Despite the integration of advanced loss functions designed to address class imbalance and boundary refinement, their immediate impact is not clearly visible in these initial results. This could stem from:
    *   **Insufficient Training**: 5 epochs may not be enough for the model to fully learn from these complex loss signals.
    *   **Suboptimal Loss Weights**: The current weights for Tversky, Focal, and Boundary losses (all set to 1.0) might not be ideal and may require fine-tuning.
    *   **Simplified Decoder Architecture**: The current U-Net style decoder, while an improvement over the initial simple ConvTranspose, might still be too simplistic to generate highly precise masks from the ViT backbone's patch embeddings.

## Propose Refinement Strategies

### Subtask:
Suggest concrete next steps for further refining the segmentation performance. This could include adjusting loss weights, exploring post-processing techniques like DenseCRF, or considering architectural modifications.


### Proposed Refinement Strategies for Segmentation Performance

Based on the qualitative analysis, several concrete steps can be taken to refine the segmentation performance, particularly focusing on leveraging the newly integrated advanced loss functions and addressing the identified issues:

1.  **Optimize Loss Function Weights:**
    *   **Strategy**: Systematically tune the weights (e.g., `tversky_weight`, `focal_weight`, `boundary_weight`) within the `CombinedSegmentationLoss`.
    *   **Rationale**: The current equal weighting (1.0 for each) might not be optimal. Boundary Loss, for instance, might need a higher weight to enforce sharper edges, while Focal Loss could be adjusted to better handle the class imbalance. This can be done through a grid search, random search, or more advanced optimization techniques.
    *   **Method**: Experiment with different combinations, for example:
        *   Emphasize boundary: `tversky_weight=0.5, focal_weight=0.5, boundary_weight=2.0`
        *   Emphasize imbalance: `tversky_weight=1.0, focal_weight=2.0, boundary_weight=1.0`
        *   A good starting point would be to slightly increase the boundary loss weight to address the blurry edges.

2.  **Increase Training Epochs:**
    *   **Strategy**: Extend the training duration beyond the initial 5 epochs.
    *   **Rationale**: Complex loss functions like Tversky, Focal, and Boundary Loss require more iterations to effectively learn intricate patterns, especially given the multi-task nature of the model and the complexity of medical image segmentation. 5 epochs is generally insufficient for a deep learning model to converge to an optimal solution for such a task.
    *   **Method**: Run training for 20-50 epochs or until validation loss plateaus, ensuring proper early stopping mechanisms are in place if performance degrades.

3.  **Enhance Segmentation Decoder Architecture:**
    *   **Strategy**: Further refine the U-Net style decoder, potentially incorporating skip connections from earlier layers of the ViT backbone.
    *   **Rationale**: While the current decoder is an improvement, highly precise segmentation often benefits from combining high-level semantic features (from deep layers) with low-level spatial features (from shallow layers) to preserve fine-grained details.
    *   **Method**: Explore architectures that include more sophisticated upsampling paths or incorporate attention mechanisms within the decoder to better guide feature fusion.

4.  **Implement Post-processing Techniques:**
    *   **Strategy**: Apply post-processing to the predicted masks to refine boundaries and reduce noise.
    *   **Rationale**: Even with improved loss functions, model predictions can be noisy or lack perfect topological consistency. Techniques like Conditional Random Fields (CRF) or DenseCRF can enforce spatial smoothness and better align segmentations with image edges.
    *   **Method**: After obtaining `sigmoid(seg_output)`, apply a post-processing step (e.g., using `pydensecrf` library) to refine the predicted mask before evaluation.

5.  **Data Augmentation (Advanced):**
    *   **Strategy**: Implement more aggressive or domain-specific data augmentation techniques.
    *   **Rationale**: Medical images can be highly variable. Augmentations like random rotations, flips, shifts, brightness adjustments, and elastic deformations can make the model more robust and generalizable.
    *   **Method**: Integrate `albumentations` or similar libraries into the `BrainTumorDataset`'s `transform` pipeline.

6.  **Analyze False Positives/Negatives Systematically:**
    *   **Strategy**: Perform a more quantitative analysis of false positives and false negatives.
    *   **Rationale**: Understanding *where* and *why* the model fails will provide targeted insights for improvement. This could involve examining cases of under-segmentation (high false negatives for large tumors) or over-segmentation (false positives in healthy tissue).
    *   **Method**: Implement metrics like False Positive Rate and False Negative Rate specifically for segmentation and sort validation samples by these error types.

## Final Task

### Subtask:
Provide a comprehensive summary of the qualitative observations, including both successes and challenges, and detail the proposed next steps for model improvement.


## Calculate Dice Similarity Coefficient (DSC)

In [ ]:
import torch

def dice_score(y_pred_logits: torch.Tensor, y_true: torch.Tensor, threshold: float = 0.5) -> float:
    """
    Calculates the Dice Similarity Coefficient (DSC) for segmentation masks.

    Args:
        y_pred_logits (torch.Tensor): Predicted segmentation logits from the model.
                                     Shape (N, C, H, W) or (N, H, W) for binary.
        y_true (torch.Tensor): Ground truth segmentation masks. Shape (N, C, H, W) or (N, H, W) for binary.
        threshold (float): Threshold to convert predicted probabilities to binary masks.

    Returns:
        float: The mean Dice score across the batch.
    """
    # Apply sigmoid to convert logits to probabilities, then threshold to get binary prediction
    y_pred_proba = torch.sigmoid(y_pred_logits)
    y_pred_binary = (y_pred_proba > threshold).float()

    # Flatten tensors
    y_pred_flat = y_pred_binary.view(-1)
    y_true_flat = y_true.view(-1).float() # Ensure y_true is float

    # Calculate intersection and union
    intersection = (y_pred_flat * y_true_flat).sum()
    total_sum = y_pred_flat.sum() + y_true_flat.sum()

    # Handle case where both are empty (perfect score = 1.0)
    if total_sum == 0:
        return 1.0

    dice = (2. * intersection + 1e-8) / (total_sum + 1e-8) # Add small epsilon for numerical stability
    return dice.item()

print("Dice Similarity Coefficient function defined.")

### Update Training Loop for DSC Calculation